# NB_Dynamic_Alerts_SilverToGold
**Source:** `AI_Medallion.Silver.dynamic_alerts`
**Target:** `AI_Medallion.Gold.dim_dynamic_alerts`
**Grain:** One row per `NotificationID`
**Load type:** Overwrite (direct rename, no business logic)

In [1]:
# Workspace folder: C:\\spark-dev\\CoE_transformation_framework (see README)
import os
from pathlib import Path
os.environ["HADOOP_HOME"] = r"C:\hadoop\hadoop-3.5.0"
os.environ["HADOOP_PREFIX"] = os.environ["HADOOP_HOME"]
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]
 
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("HADOOP_PREFIX:", os.environ["HADOOP_PREFIX"])
print("winutils exists:", os.path.isfile(
    os.path.join(os.environ["HADOOP_HOME"], "bin", "winutils.exe")
))
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
REBUILD_MEDALLION_CACHE = True  # True: rebuild; False: use cache
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
from Common.io import sync_rebuild_medallion_cache_env

sync_rebuild_medallion_cache_env(REBUILD_MEDALLION_CACHE)

HADOOP_HOME: C:\hadoop\hadoop-3.5.0
HADOOP_PREFIX: C:\hadoop\hadoop-3.5.0
winutils exists: True


In [2]:
%run ./Common/bootstrap

Bootstrap OK: runtime=local mode=created spark=3.5.4 pyspark=3.5.4 path_reads=True onelake_read=deltalake-scan
Repo root: C:\spark-dev\CoE_Transformation_Framework_copilot
spark.jars.packages=io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-azure:3.3.4,com.azure:azure-storage-blob:12.25.1
OneLake reads: deltalake scan() + az login -> Spark (column mapping; avoids local PySpark abfss hang).
TABLE_CURSOR_BRONZE -> abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1ee46463-2495-44f5-9dc9-d635d0067483/Tables/Bronze/cursor_usage
Tip: set MEDALLION_DEBUG_READ=1 before bootstrap to log read progress.
Tip: large tables — read_table() one at a time; transient Azure errors auto-retry.
Local table cache ON (default): parquet under ~/.fabric/medallion_cache/. Set REBUILD_MEDALLION_CACHE = True in settings to force OneLake. reload_io_helpers(globals()) after git pull — no kernel restart.
OneLake writes: deltalake + az login (merge/overwrite). Set MEDALLION_ONELAK

In [3]:
# TODO: move these two constants to Common/config.py if not defined there yet
TABLE_DYNAMIC_ALERTS_SILVER = globals().get(
    "TABLE_DYNAMIC_ALERTS_SILVER", "AI_Medallion.Silver.dynamic_alerts"
)
TABLE_DIM_DYNAMIC_ALERTS = globals().get(
    "TABLE_DIM_DYNAMIC_ALERTS", "AI_Medallion.Gold.dim_dynamic_alerts"
)

silver = read_table_broken_lineage(TABLE_DYNAMIC_ALERTS_SILVER, "dynamic_alerts_silver")

dim = (
    silver.select(
        F.col("dynamic_alerts_notification_id").alias("NotificationID"),
        F.col("dynamic_alerts_notification_type").alias("NotificationType"),
        F.col("dynamic_alerts_description").alias("Description"),
        F.col("dynamic_alerts_is_active").alias("IsActive"),
        F.col("dynamic_alerts_publication_date").cast("date").alias("PublicationDate"),
        F.col("dynamic_alerts_severity").alias("Severity"),
    )
    .filter(F.col("NotificationID").isNotNull())
    .dropDuplicates(["NotificationID"])
)

dim.printSchema()
dim.orderBy("NotificationID").show(truncate=False)
print("rows:", dim.count(), "| distinct ids:", dim.select("NotificationID").distinct().count())

read_table_broken_lineage: scan -> local cache (AI_Medallion.Silver.dynamic_alerts)
read_table: deltalake scan abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1e… (pass 1/6)
read_table: OneLake object-store concurrency limit=1 (raise MEDALLION_OBJECT_STORE_CONCURRENCY if reads are too slow)
read_table: deltalake scan() streaming batches…
read_table: scan streaming… 2 rows from OneLake so far
read_table: scan finished (4 rows)
read_table: opening local parquet cache (4 rows) -> C:\Users\soukelkr\.fabric\medallion_cache\AI_Medallion__Silver__dynamic_alerts
read_table_broken_lineage: opened cache after scan (dynamic_alerts_silver)
root
 |-- NotificationID: string (nullable = true)
 |-- NotificationType: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- IsActive: string (nullable = true)
 |-- PublicationDate: date (nullable = true)
 |-- Severity: string (nullable = true)

+--------------+----------------+---------------------------------

In [5]:
apply_local_merge_spark_conf(spark)
 
write_full_table(dim, TABLE_DIM_DYNAMIC_ALERTS)
 
show_sample(dim.limit(10))

publish_merge_staging: AI_Medallion.Gold.dim_dynamic_alerts -> C:\Users\soukelkr\.fabric\merge_staging\AI_Medallion__Gold__dim_dynamic_alerts


materialize staging: Spark JVM parquet (8 partitions)
deltalake write mode=overwrite abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1ee46463-2495-44f5-9dc9-d635d0067483/Tables/Gold/dim_dynamic_alerts (local staging -> OneLake)...
deltalake write chunk 1 (1 rows) mode=overwrite
deltalake write chunk 2 (1 rows) mode=append
deltalake write chunk 3 (2 rows) mode=append
deltalake write done


DataFrame[NotificationID: string, NotificationType: string, Description: string, IsActive: string, PublicationDate: date, Severity: string]

In [ ]:
apply_local_merge_spark_conf(spark)

# TODO: replace with the overwrite helper that exists in Common/io.py
# (check what NB_CursorOnboard_Gold's publish_merge_staging / merge_incremental call underneath)
overwrite_table(spark, dim, TABLE_DIM_DYNAMIC_ALERTS)

show_sample(dim.limit(10))